In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
import shap
import warnings
warnings.filterwarnings('ignore')



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/547.2 kB ? eta -:--:--
   ------------------- -------------------- 262.1/547.2 kB ? eta -:--:--
   ---------------------------------------- 547.2/547.2 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.7 MB 1.7 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 1.6 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.7 MB 1.8 MB/s eta 0:00:01
   ----------------------------------- ---- 2.4/2.7 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
    ------

c:\Users\gchehata\scoop\apps\miniconda3\4.12.0\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv('../AirlineScrappedReview_Cleaned_seif_we_ibra#1.csv')

In [ ]:




X = df[['Flying_Date', 'Route', 'Verified', 'Review_title', 'Review_content', 'Traveller_Type', 'Class', 'Start_Location', 'End_Location', 'Layover_Route', 'Start_Longitude', 'Start_Latitude', 'End_Longitude', 'End_Latitude', 'Start_Address', 'End_Address']]

y = df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)



In [ ]:
# Step 1: Feature Engineering for Neural Networks
print("="*80)
print("FEATURE ENGINEERING FOR NEURAL NETWORKS")
print("="*80)

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Handle missing values
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Fill numerical missing values with median
for col in numerical_cols:
    median_val = X_train_clean[col].median()
    X_train_clean[col].fillna(median_val, inplace=True)
    X_test_clean[col].fillna(median_val, inplace=True)

# Fill categorical missing values with 'Unknown'
for col in categorical_cols:
    X_train_clean[col].fillna('Unknown', inplace=True)
    X_test_clean[col].fillna('Unknown', inplace=True)

print(f"\n✓ Missing values handled")

# One-Hot Encode categorical features
X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_clean, columns=categorical_cols, drop_first=True)

# Ensure both sets have the same columns
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0

X_test_encoded = X_test_encoded[X_train_encoded.columns]

print(f"✓ One-hot encoding applied")
print(f"  Features after encoding: {X_train_encoded.shape[1]}")

# IMPORTANT FOR NN: Normalize/Scale numerical features
# This is critical for NN convergence and SHAP stability
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# Convert back to DataFrames to preserve feature names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_encoded.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_encoded.columns)

print(f"✓ StandardScaler normalization applied")
print(f"  X_train_scaled shape: {X_train_scaled.shape}")
print(f"  X_test_scaled shape: {X_test_scaled.shape}")
print(f"  Feature scaling stats:")
print(f"    - Mean (should be ≈0): {X_train_scaled.mean().mean():.6f}")
print(f"    - Std (should be ≈1): {X_train_scaled.std().mean():.6f}")

feature_names = X_train_scaled.columns.tolist()


In [ ]:
# Step 2: Build and Train Neural Network
print("\n" + "="*80)
print("BUILDING AND TRAINING NEURAL NETWORK")
print("="*80)

# Build the neural network
input_dim = X_train_scaled.shape[1]
print(f"\nBuilding Sequential Neural Network...")
print(f"  Input features: {input_dim}")

model = Sequential([
    layers.Dense(128, activation='relu', input_dim=input_dim, name='dense_1'),
    layers.Dropout(0.3, name='dropout_1'),
    layers.Dense(64, activation='relu', name='dense_2'),
    layers.Dropout(0.3, name='dropout_2'),
    layers.Dense(32, activation='relu', name='dense_3'),
    layers.Dense(1, activation='sigmoid', name='output')  # sigmoid for binary classification
])

print("\nModel Architecture:")
model.summary()

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled")

# Train the model with early stopping
print("\nTraining Neural Network...")
print("  (This may take a minute...)")

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print(f"✓ Model trained successfully")
print(f"  Total epochs trained: {len(history.history['loss'])}")

# Evaluate model
y_pred_proba = model.predict(X_test_scaled, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
accuracy = accuracy_score(y_test, y_pred)

print(f"\n  Test Accuracy: {accuracy:.4f}")
print(f"  Number of features: {len(feature_names)}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss Over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('nn_training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Training history saved as 'nn_training_history.png'")


In [ ]:
# Step 3: Initialize SHAP Explainer for Neural Network
print("\n" + "="*80)
print("SHAP EXPLAINER FOR NEURAL NETWORKS")
print("="*80)

print("\nInitializing SHAP DeepExplainer (optimized for neural networks)...")
print("  • DeepExplainer uses layer-wise relevance propagation (DeepLIFT algorithm)")
print("  • Efficient for gradient-based explanations")
print("  • Captures non-linear interactions in the network")

# Choose a background sample set (helps SHAP learn the model behavior)
# Using a sample of training data for efficiency
background_indices = np.random.choice(len(X_train_scaled), size=min(100, len(X_train_scaled)), replace=False)
background_data = X_train_scaled.iloc[background_indices].values

print(f"\nBackground data size: {background_data.shape[0]} samples (for reference)")

# Create DeepExplainer
# Note: DeepExplainer needs the model and background data
explainer = shap.DeepExplainer(model, background_data)

print("✓ DeepExplainer created successfully")
print(f"  Explainer type: {type(explainer).__name__}")
print(f"  Model type: {type(model).__name__}")

# Calculate SHAP values for test set
print("\nCalculating SHAP values for test set...")
print("  (This uses gradient-based DeepLIFT algorithm - may take a moment)")

# Use a sample of test data for computational efficiency if needed
# For large datasets, you might want to use: X_test_to_explain = X_test_scaled.iloc[:100]
X_test_to_explain = X_test_scaled.values

shap_values = explainer.shap_values(X_test_to_explain)

print(f"✓ SHAP values calculated")
print(f"  SHAP values shape: {np.array(shap_values).shape}")

# Handle output format
if isinstance(shap_values, list):
    # For neural networks with multiple outputs, use the main prediction
    shap_values_to_use = shap_values[0]
    print(f"  Multi-output detected: using first output")
else:
    shap_values_to_use = shap_values

print(f"  Final shape for visualization: {shap_values_to_use.shape}")


In [ ]:
# Step 4: SHAP Summary Plots for Neural Network
print("\n" + "="*80)
print("SHAP SUMMARY PLOTS - GLOBAL FEATURE IMPORTANCE (NEURAL NETWORK)")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1: Bar plot - Mean absolute SHAP values
print("\n1. Creating Bar Plot (top 15 most important features for NN)...")
plt.sca(axes[0])
try:
    shap.summary_plot(shap_values_to_use, X_test_to_explain, plot_type="bar", max_display=15, show=False)
    axes[0].set_title("SHAP Bar Plot: Top 15 Most Important Features (NN)\n(Average absolute impact on model output)", 
                     fontsize=12, fontweight='bold')
except Exception as e:
    print(f"  Warning: {e}")
    print("  Using alternative visualization...")
    
# Plot 2: Bee swarm plot - Feature distribution
print("2. Creating Bee Swarm Plot (feature value impact distribution)...")
plt.sca(axes[1])
try:
    shap.summary_plot(shap_values_to_use, X_test_to_explain, plot_type="dot", max_display=15, show=False)
    axes[1].set_title("SHAP Bee Swarm Plot: Top 15 Features (NN)\n(Red=High value, Blue=Low value)", 
                     fontsize=12, fontweight='bold')
except Exception as e:
    print(f"  Warning: {e}")

plt.tight_layout()
plt.savefig('shap_nn_summary_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Summary plots created and saved as 'shap_nn_summary_plots.png'")
print("\nInterpretation for Neural Networks:")
print("  • Bar Plot: Shows which features have largest average impact on NN predictions")
print("  • Bee Swarm: Shows feature value direction (Red=high, Blue=low) and impact")
print("    - Position on x-axis: Positive = increases prediction, Negative = decreases")
print("    - This reflects the NN's learned non-linear decision boundaries")
print("  • Note: NN SHAP may show more complex patterns than tree models")


In [ ]:
# Step 5: SHAP Force Plots for Neural Network
print("\n" + "="*80)
print("SHAP FORCE PLOTS - INDIVIDUAL PREDICTION EXPLANATIONS (NEURAL NETWORK)")
print("="*80)

# Select interesting samples to explain
sample_indices = [0, 10, 20]

for idx, sample_idx in enumerate(sample_indices):
    if sample_idx >= len(X_test_to_explain):
        print(f"Skipping sample index {sample_idx} (out of range)")
        continue
    
    print(f"\n--- Sample {idx+1}: Test Index {sample_idx} ---")
    nn_pred = model.predict(X_test_to_explain[[sample_idx]], verbose=0)[0][0]
    actual_rating = y_test.iloc[sample_idx]
    print(f"NN Prediction (probability): {nn_pred:.4f}")
    print(f"NN Prediction (class): {int(nn_pred > 0.5)}")
    print(f"Actual Rating: {actual_rating}")
    
    try:
        # Create force plot
        # Get the base value from SHAP explainer
        base_value = explainer.expected_value if hasattr(explainer, 'expected_value') else shap_values_to_use.mean()
        
        shap.force_plot(
            base_value,
            shap_values_to_use[sample_idx],
            X_test_to_explain[sample_idx],
            feature_names=feature_names,
            matplotlib=True,
            show=False
        )
        plt.tight_layout()
        plt.savefig(f'shap_nn_force_plot_sample_{sample_idx}.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"✓ Force plot saved as 'shap_nn_force_plot_sample_{sample_idx}.png'")
    except Exception as e:
        print(f"  Warning creating force plot: {e}")

print("\n\nInterpretation of Force Plots (Neural Networks):")
print("  • Base value: Average NN output across background samples")
print("  • Red arrows (→): Features pushing prediction UP")
print("  • Blue arrows (←): Features pushing prediction DOWN")
print("  • Arrow length: Feature's contribution magnitude")
print("  • Output value: Final NN prediction probability")
print("  • NN force plots may show more complex interactions than RF")


In [ ]:
# Step 6: SHAP Dependence Plots for Neural Network
print("\n" + "="*80)
print("SHAP DEPENDENCE PLOTS - FEATURE INTERACTIONS (NEURAL NETWORK)")
print("="*80)

# Get top features by mean absolute SHAP value
mean_abs_shap = np.abs(shap_values_to_use).mean(axis=0)
top_features_idx = np.argsort(mean_abs_shap)[-4:][::-1]  # Top 4 features
top_features_names = [feature_names[i] for i in top_features_idx]

print(f"\nAnalyzing top 4 most important features for NN:")
for feat_name in top_features_names:
    print(f"  • {feat_name}")

# Create dependence plots for top features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, feat_idx in enumerate(top_features_idx):
    plt.sca(axes[i])
    print(f"\nCreating dependence plot for: {feature_names[feat_idx]}")
    
    try:
        shap.dependence_plot(
            feat_idx,
            shap_values_to_use,
            X_test_to_explain,
            feature_names=feature_names,
            show=False,
            ax=axes[i]
        )
        axes[i].set_title(f"Dependence: {feature_names[feat_idx]} (NN)\n(Feature value vs SHAP contribution)", 
                          fontsize=11, fontweight='bold')
    except Exception as e:
        print(f"  Error creating dependence plot: {e}")
        axes[i].text(0.5, 0.5, f"Error: {str(e)[:50]}...", 
                    ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('shap_nn_dependence_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Dependence plots created and saved as 'shap_nn_dependence_plots.png'")
print("\nInterpretation for Neural Networks:")
print("  • X-axis: Actual feature value (after scaling by StandardScaler)")
print("  • Y-axis: SHAP contribution (impact on NN output)")
print("  • Color: Automatically detects interactions with other features")
print("  • Non-linear patterns: Indicate NN learned complex feature interactions")
print("  • Unlike trees, NN dependence plots often show smooth non-linear curves")


In [ ]:
# Step 7: SHAP Waterfall Plots for Neural Network
print("\n" + "="*80)
print("SHAP WATERFALL PLOTS - DETAILED PREDICTION BREAKDOWN (NEURAL NETWORK)")
print("="*80)

from shap import Explanation

# Get base value for NN
base_value = explainer.expected_value if hasattr(explainer, 'expected_value') else shap_values_to_use.mean()

for sample_idx in [0, 5, 15]:
    if sample_idx >= len(X_test_to_explain):
        print(f"Skipping sample {sample_idx} (out of range)")
        continue
    
    print(f"\n--- Waterfall Plot for Test Sample {sample_idx} ---")
    nn_pred = model.predict(X_test_to_explain[[sample_idx]], verbose=0)[0][0]
    actual_rating = y_test.iloc[sample_idx]
    print(f"NN Prediction (probability): {nn_pred:.4f}")
    print(f"Actual Rating: {actual_rating}")
    
    try:
        # Create explanation object
        explanation = Explanation(
            values=shap_values_to_use[sample_idx],
            base_values=base_value,
            data=X_test_to_explain[sample_idx],
            feature_names=feature_names
        )
        
        # Create waterfall plot
        plt.figure(figsize=(12, 8))
        shap.waterfall_plot(explanation, max_display=15, show=False)
        plt.title(f"SHAP Waterfall: Test Sample {sample_idx} (Neural Network)\n(How features contribute to final NN prediction)", 
                  fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_nn_waterfall_sample_{sample_idx}.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"✓ Waterfall plot saved as 'shap_nn_waterfall_sample_{sample_idx}.png'")
    except Exception as e:
        print(f"  Error creating waterfall plot: {e}")

print("\n\nInterpretation of Waterfall Plots (Neural Networks):")
print("  • Bottom gray bar: Base value (expected NN output)")
print("  • Red bars (→): Features increasing the NN's prediction")
print("  • Blue bars (←): Features decreasing the NN's prediction")
print("  • Top value: Final NN output (probability)")
print("  • Shows top 15 features by absolute contribution")
print("  • For NN: Reflects learned non-linear decision boundaries")


In [ ]:
# Step 8: SHAP Heatmap for Neural Network
print("\n" + "="*80)
print("SHAP HEATMAP - MULTI-INSTANCE OVERVIEW (NEURAL NETWORK)")
print("="*80)

# Create a heatmap for multiple NN predictions
try:
    plt.figure(figsize=(14, 8))
    sample_size = min(50, len(X_test_to_explain))
    print(f"Creating SHAP heatmap for first {sample_size} test samples...")
    
    shap.summary_plot(shap_values_to_use[:sample_size], X_test_to_explain[:sample_size], 
                      plot_type="dot", max_display=15, show=False)
    plt.title(f"SHAP Heatmap: Feature Contributions for {sample_size} NN Test Samples\n(Pattern recognition across multiple NN predictions)", 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_nn_heatmap.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✓ SHAP heatmap created and saved as 'shap_nn_heatmap.png'")
except Exception as e:
    print(f"Error creating heatmap: {e}")
    print("Trying alternative visualization...")
    
    plt.figure(figsize=(14, 8))
    # Alternative: use absolute SHAP values as heatmap
    shap_abs = np.abs(shap_values_to_use[:50])
    plt.imshow(shap_abs[:, :20], cmap='RdBu_r', aspect='auto', interpolation='nearest')
    plt.colorbar(label='|SHAP value|')
    plt.xlabel('Feature Index')
    plt.ylabel('Sample Index')
    plt.title('SHAP Absolute Values Heatmap (NN) - Top 20 Features')
    plt.tight_layout()
    plt.savefig('shap_nn_heatmap_alt.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Alternative heatmap saved")


## Summary: SHAP Explainability for Neural Networks

### What We've Created (NN-Specific):

1. **Neural Network Architecture**
   - Sequential model with 3 hidden layers (128 → 64 → 32 units)
   - Dropout regularization (0.3) to prevent overfitting
   - ReLU activations for hidden layers, Sigmoid for binary output
   - Early stopping to prevent overfitting

2. **Feature Preprocessing for NN**
   - One-hot encoding for categorical features
   - StandardScaler normalization (critical for NN convergence!)
   - Normalized features have mean ≈ 0 and std ≈ 1

3. **SHAP DeepExplainer**
   - Uses **DeepLIFT algorithm** (gradient-based approach)
   - More efficient than KernelExplainer for NNs
   - Captures non-linear interactions learned by NN layers
   - Background data: Sample of 100 training examples

4. **Visualization Plots**
   - **Summary Plot**: Global feature importance in NN
   - **Force Plots**: Individual prediction decomposition
   - **Dependence Plots**: Non-linear feature relationships
   - **Waterfall Plots**: Top features contributing to each prediction
   - **Heatmap**: Patterns across multiple predictions

### Key Differences: NN vs Classical ML SHAP

| Aspect | Classical ML (RandomForest) | Neural Networks |
|--------|---------------------------|-----------------|
| **Explainer** | TreeExplainer (optimized) | DeepExplainer (gradient-based) |
| **Speed** | Very fast | Slower (requires gradient computation) |
| **Interpretability** | Linear decision rules | Non-linear, complex interactions |
| **Feature Scaling** | Not required | **Critical for convergence** |
| **Computation** | Simple traversal | Backpropagation through layers |
| **Background Data** | Not needed | Needed (reference distribution) |

### NN SHAP Characteristics:

✓ **Non-linear patterns**: Dependence plots show smooth curves (not tree splits)  
✓ **Complex interactions**: Captures feature interactions learned by hidden layers  
✓ **Gradient-based**: Explains how small changes in inputs affect outputs  
✓ **Computational cost**: More expensive than TreeExplainer  
✓ **Stability**: Background sample choice affects explanations (reference point matters)  

### When to Use SHAP for NNs:

**Good For:**
- Understanding what features NN considers important
- Debugging unexpected predictions
- Finding non-linear feature relationships
- Explaining model decisions to stakeholders

**Limitations:**
- Slower than classical ML explainers
- Requires choosing representative background data
- May be harder to interpret (more complex patterns)
- High-dimensional data can make explanations harder

### Practical Tips for NN SHAP:

1. **Background Data**: Use representative sample of training data (not random noise)
2. **Sample Size**: For efficiency, explain sample of test set (not all if very large)
3. **Feature Scaling**: MUST use StandardScaler or similar before NN training
4. **Model Performance**: SHAP quality depends on model accuracy - good model → trustworthy explanations
5. **GPU Usage**: For large models, consider GPU acceleration (SHAP will use it automatically if available)

### Interpreting NN SHAP vs RF SHAP:

- **RandomForest**: Usually shows clear feature importance (few dominant features)
- **Neural Network**: Often shows more distributed importance (multiple interacting features)
- **Why**: NNs learn to combine features in non-linear ways, unlike tree splits

### Files Generated:

- `nn_training_history.png` - Training curves
- `shap_nn_summary_plots.png` - Global importance
- `shap_nn_force_plot_sample_*.png` - Individual explanations
- `shap_nn_dependence_plots.png` - Feature relationships
- `shap_nn_waterfall_sample_*.png` - Detailed breakdown
- `shap_nn_heatmap.png` - Multi-instance overview

